# Computational Theory: SHA-256

## Problem 1: Representing SHA-256 Data

### 32-bit words
Before beginning with any proper SHA-256 implementation starting with looking into how the input, output and intermediate data can be represented in Python.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §2.1: defines a word as a group of 32 bits for SHA-256.
- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §3.2: words are added modulo 2<sup>32</sup>, which is why the sum below is masked back to 32 bits.

In [1]:
# 32-bit words.

# largest 32-bit value.
mask = 0xFFFFFFFF

# The largest unsigned 32-bit value, and 1 to add to it.
hello = 0xFFFFFFFF
bye = 1

# The sum is 0x100000000, which needs 33 bits. ANDing it with the mask
# drops bit 32, so the result wraps to 0.
result = (hello + bye) & mask

# Printing value as hex.
print(f"0x{result:08X}")

0x00000000


### Sequences of 32-bit words

SHA-256 works with 32 bit words the whole way through. Everything it uses comes in fixed size groups of them: the 16 words of a message block, the 64-word message schedule, the 64 round constants, and the 8 words of the hash state.

I'll represent each of these as a Python list of ints. Every element is an unlimited size
integer, so each one has to be masked manually to stay within 32 bits.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §5.2.1: each 512-bit message block is split into sixteen 32-bit words.
- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.2.2: the sixty-four 32-bit round constants.
- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §5.3.3: the initial hash value, made of eight 32-bit words.
- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §6.2.2: the 64-word message schedule and the hash computation.

In [2]:
# Example of eight random 32 bit words.
words = [
    0x1a2b3c4d, 0xdeadbeef, 0x00000001, 0xffffffff,
    0x80000000, 0x12345678, 0x26d8aabe, 0x0badf00d,
]

# Add each word to itself, masking so we keep 32 bits.
doubled = [(w + w) & mask for w in words]

# Print each value as hex
print([f"{w:08x}" for w in doubled])

['3456789a', 'bd5b7dde', '00000002', 'fffffffe', '00000000', '2468acf0', '4db1557c', '175be01a']


You can use a Python list of ints to represent sequences of 32 bit words, but since Python ints can grow large, you have to apply a mask after every operation to keep the values within 32 bits.

### Input Messages

SHA-256 takes a message as a sequence of bytes, I'll use Python bytes objects to represent input messages. Text has to be encoded first. UTF-8 turns ASCII characters into one byte each and other characters into several bytes.

**References**

- [Python docs: `str.encode`](https://docs.python.org/3/library/stdtypes.html#str.encode): converts a string to bytes using a given encoding.
- [RFC 3629: UTF-8](https://www.rfc-editor.org/rfc/rfc3629): the UTF-8 standard, which encodes ASCII characters as one byte and other characters as two to four bytes.

In [3]:
# Encode text messages as UTF-8 bytes.
ascii_msg = "hello".encode("utf-8")
# "é" is not an ASCII character, so UTF-8 encodes it as two bytes.
non_ascii_msg = "café".encode("utf-8")

# Show each message's bytes in hex, then compare how many characters it has
# with how many bytes (and bits) it takes up.
for msg in [ascii_msg, non_ascii_msg]:
    chars = len(msg.decode("utf-8"))
    print(msg, msg.hex(), f"{chars} characters, {len(msg)} bytes, {len(msg) * 8} bits")

b'hello' 68656c6c6f 5 characters, 5 bytes, 40 bits
b'caf\xc3\xa9' 636166c3a9 4 characters, 5 bytes, 40 bits


bytes is the right fit because SHA-256 hashes bytes and not characters. It's
immutable, its length in bits is just `len(msg) * 8`, and it's what `hashlib.sha256` accepts, so I can test against it.

**References**

- [Python docs: `bytes`](https://docs.python.org/3/library/stdtypes.html#bytes): Python's immutable sequence of bytes.
- [Python docs: `hashlib`](https://docs.python.org/3/library/hashlib.html): Python's built-in SHA-256, used to check results.

### 512-bit message blocks

SHA-256 processes a message in 512 bit blocks. That's 64 bytes, or 16
32-bit words. I can represent a block as a list of 16 ints, built by splitting
64 bytes into 4 byte chunks and converting each chunk to a word.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §5.2.1: the padded message is parsed into 512-bit blocks of sixteen 32-bit words.

In [4]:
# A 64 byte block
block = bytes(range(64))

# Split into 4 byte chunks and then we convert each one to a 32 bit word.
block_words = [int.from_bytes(block[i:i + 4], "big") for i in range(0, 64, 4)]

print(f"{len(block) * 8} bits, {len(block_words)} words")
print([f"{w:08x}" for w in block_words])

512 bits, 16 words
['00010203', '04050607', '08090a0b', '0c0d0e0f', '10111213', '14151617', '18191a1b', '1c1d1e1f', '20212223', '24252627', '28292a2b', '2c2d2e2f', '30313233', '34353637', '38393a3b', '3c3d3e3f']


A list of 16 words matches how the standard describes a block. Real messages are almost never a multiple of 64 bytes.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §5.1.1: how a message is padded to a multiple of 512 bits before it is split into blocks.

### The final 256-bit hash value

SHA-256 ends with eight 32 bit words of hash state. Put
together, they make up the 256-bit digest. I'll keep it as a list of 8 words while
computing, then can return it as a 64 character lowercase hex string.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §6.2.2: the final digest is the eight hash words H<sub>0</sub> to H<sub>7</sub> joined together.
- [NIST SHA-256 example values](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA256.pdf): the worked example for `"abc"` that the expected digest below comes from.

In [5]:
# Final hash state for the message "abc".
final_words = [
    0xba7816bf, 0x8f01cfea, 0x414140de, 0x5dae2223,
    0xb00361a3, 0x96177a9c, 0xb410ff61, 0xf20015ad,
]

# Join the words as 8 digit hex strings so we can get the 64 character digest.
digest = "".join(f"{w:08x}" for w in final_words)

print(digest, f"({len(digest)} hex characters)")

# Expected SHA-256 digest of 'abc' from the standards examples.
assert digest == "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad"

ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad (64 hex characters)


Eight words are handy while computing, since each one gets updated separately.
A hex string is also the standard way to show a digest since each hex digit is 4 bits, so
256 bits is always 64 characters.

### Big-endian integers

SHA-256 reads and writes words in big-endian order, the most significant byte
comes first. Python's `int.from_bytes` and
`int.to_bytes` take the byte order as an argument, so I can easily pass `"big"`.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §3.1: the standard uses the big-endian convention for words.
- [Python docs: `int.from_bytes`](https://docs.python.org/3/library/stdtypes.html#int.from_bytes) and [`int.to_bytes`](https://docs.python.org/3/library/stdtypes.html#int.to_bytes): convert between bytes and ints with a chosen byte order.

In [6]:
# Four bytes in order: 0x01, 0x02, 0x03, 0x04.
chunk = bytes([0x01, 0x02, 0x03, 0x04])

# Read the same bytes both ways.
big = int.from_bytes(chunk, "big")
little = int.from_bytes(chunk, "little")

print(f"big-endian:    0x{big:08x}")
print(f"little-endian: 0x{little:08x}")

# Convert back with the same byte order and you get the original bytes.
assert big.to_bytes(4, "big") == chunk

big-endian:    0x01020304
little-endian: 0x04030201


Byte order matters since the same bytes can give you different words. Most
computers are little-endian, so the order has to be stated directly.
If you read a message with the wrong order then you'd get a hash that looks fine but is
completely different.

## Problem 2: SHA-256 Bitwise Operations

SHA-256 applies six bitwise functions to each message block. All of them operate on 32-bit words, so I'll use NumPy's uint32 type to keep values at that size.

The functions below follow the definitions in FIPS 180-4. ROTR stands for rotate right, meaning bits shifted off the right side wrap around to the left. A normal right shift (>>) doesn't wrap, so the rotation combines a right shift with a left shift.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.1.2: defines the SHA-256 logical functions.
- [NumPy docs: `numpy.uint32`](https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint32): the unsigned 32-bit integer type.

### Choice function: Ch(x, y, z)

The choice function uses x to decide which bits to take from y or z. If a bit in x is 1 it takes the matching bit from y. If the bit is 0 it takes the bit from z.

Defined as:

$$\text{Ch}(x, y, z) = (x \land y) \oplus (\lnot x \land z)$$

Here $\land$ is bitwise AND (`&`), $\lnot$ is bitwise NOT (`~`) and $\oplus$ is bitwise XOR (`^`).

Implementation:

In [7]:
import numpy as np  # NumPy gives us the uint32 type for 32-bit words

def Ch(x, y, z):
    """SHA-256 choice function for three 32-bit words."""
    # Force all three inputs to 32 bits so the bitwise math matches SHA-256
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    # For each bit, take from y where x is 1 and from z where x is 0
    return (x & y) ^ (~x & z)

Tests:

In [8]:
# First test. x is all 1s so every bit picks from y, and the result should be y.
x = np.uint32(0xFFFFFFFF)
y = np.uint32(0x12345678)
z = np.uint32(0xABCDEF01)

# Check the result matches y
assert Ch(x, y, z) == y
# Print the result as 8 hex digits
print(f"Ch: 0x{Ch(x, y, z):08x}")

# Second test. x is all 0s so every bit picks from z, and the result should be z.
assert Ch(np.uint32(0), y, z) == z
# Print the result as 8 hex digits
print(f"Ch with x = 0: 0x{Ch(0, y, z):08x}")

Ch: 0x12345678
Ch with x = 0: 0xabcdef01


### Majority function: Maj(x, y, z)

The majority function looks at each bit position in the three inputs. If two or more of those bits are 1, the result bit is 1. If not, it's 0.

Defined as:

$$\text{Maj}(x, y, z) = (x \land y) \oplus (x \land z) \oplus (y \land z)$$

The function is symmetrical because it checks every possible pair of inputs. Changing the order of x, y and z does not change the result.

In SHA-256, Maj is used in all 64 compression rounds. It takes the three working variables a, b and c as inputs and helps compute T2.

Implementation:

In [9]:
import numpy as np  # NumPy gives us the uint32 type for 32-bit words

def Maj(x, y, z):
    """SHA-256 majority function for three 32-bit words."""
    # Force all three inputs to 32 bits so the bitwise math matches SHA-256
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    # A bit is 1 when it shows up in at least two of the three inputs
    return (x & y) ^ (x & z) ^ (y & z)

Tests:

In [10]:
# First test. All three inputs are the same, so the result should match them.
x = np.uint32(0x12345678)
# Check the result matches the shared value
assert Maj(x, x, x) == x
# Print the result as 8 hex digits
print(f"Maj: 0x{Maj(x, x, x):08x}")

# Second test. Only one input has 1 bits, so no bit can have a majority
assert Maj(0xFFFFFFFF, 0, 0) == np.uint32(0)
# Print the result as 8 hex digits
print(f"Maj with two zero inputs: 0x{Maj(0xFFFFFFFF, 0, 0):08x}")

Maj: 0x12345678
Maj with two zero inputs: 0x00000000


### Sigma 0 function: Sigma0(x)

Sigma0 takes a 32-bit input and rotates its bits right by 2, 13 and 22 positions. It then XORs the three results to get a final value.

A normal right shift drops bits off the right side. Rotation does not. Bits that leave the right side come back on the left.

Defined as:

$$\Sigma_0^{\{256\}}(x) = \text{ROTR}^{2}(x) \oplus \text{ROTR}^{13}(x) \oplus \text{ROTR}^{22}(x)$$

The ROTR helper function below does the rotation using NumPy shifts and bitwise OR.

Implementation:

In [11]:
import numpy as np  # NumPy gives us the uint32 type for 32-bit words

def ROTR(x, n):
    """Rotate a 32-bit unsigned integer right by n bit positions."""
    # Force the input to 32 bits and the shift count to a plain int
    x = np.uint32(x)
    n = int(n)

    # Shift right to drop the low bits, shift left to bring them back on top
    return np.uint32((x >> np.uint32(n)) | (x << np.uint32(32 - n)))

def Sigma0(x):
    """SHA-256 uppercase sigma 0 function."""
    # Force the input to 32 bits
    x = np.uint32(x)

    # XOR the three rotated copies of x
    return ROTR(x, 2) ^ ROTR(x, 13) ^ ROTR(x, 22)

Tests:

In [12]:
# Test the rotation helper with a value that has the top bit set.
rotated = ROTR(np.uint32(0x80000001), 1)
# Rotating right by 1 should wrap the low bit to the top and shift the rest
assert rotated == np.uint32(0xC0000000)
# Print the result as 8 hex digits
print(f"ROTR test: 0x{rotated:08x}")

# Test Sigma0 with a 32-bit input
x = np.uint32(0x12345678)
# Print the result as 8 hex digits
print(f"Sigma0: 0x{Sigma0(x):08x}")
# Check the result is still a 32-bit unsigned integer
assert isinstance(Sigma0(x), np.uint32)

ROTR test: 0xc0000000
Sigma0: 0x66146474


The function rotates the same input three times, each with a different number of positions.
The first rotation moves every bit 2 positions to the right. The second moves every bit 13 positions, and the third moves every bit 22 positions.
Using different rotation amounts means each output bit is influenced by different positions in the original input.
The three rotated values are then combined using XOR. XOR gives a 1 when an odd number of the matching input bits are 1, otherwise it gives 0.
Because rotations keep all 32 bits, no bits are lost during the individual rotations. The XOR then combines the different bit positions into one result.

### Sigma 1 function: Sigma1(x)

Sigma1 takes a 32-bit input and rotates its bits right by 6, 11 and 25 positions. It then XORs the three results to get a final 32-bit value.

Defined as:

$$\Sigma_1^{\{256\}}(x) = \text{ROTR}^{6}(x) \oplus \text{ROTR}^{11}(x) \oplus \text{ROTR}^{25}(x)$$

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.1.2, equation 4.5: defines Sigma1.
- [RFC 6234](https://www.rfc-editor.org/rfc/rfc6234), §5.1: defines the same function, which it calls BSIG1.

Implementation:

In [13]:
def Sigma1(x):
    """SHA-256 uppercase sigma 1 function."""
    # Force the input to 32 bits
    x = np.uint32(x)

    # XOR the three rotated copies of x
    return ROTR(x, 6) ^ ROTR(x, 11) ^ ROTR(x, 25)

Tests:

In [14]:
x = np.uint32(0x12345678)

# Check each rotation on its own
assert ROTR(x, 6) == np.uint32(0xE048D159)
assert ROTR(x, 11) == np.uint32(0xCF02468A)
assert ROTR(x, 25) == np.uint32(0x1A2B3C09)

# Check the final Sigma1 result
assert Sigma1(x) == np.uint32(0x3561ABDA)

# A zero input should give zero
assert Sigma1(np.uint32(0)) == np.uint32(0)

# Check the result type is still uint32
assert isinstance(Sigma1(x), np.uint32)

# Print the result as 8 hex digits
print(f"Sigma1: 0x{Sigma1(x):08x}")
print("All Sigma1 tests passed.")

Sigma1: 0x3561abda
All Sigma1 tests passed.


Where it is used. Sigma1 runs in every SHA-256 compression round. It works on the variable e and helps compute T1:

$$T_1 = h + \Sigma_1^{\{256\}}(e) + \text{Ch}(e, f, g) + K_t^{\{256\}} + W_t$$

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §6.2.2: the SHA-256 hash computation, where T1 is calculated.

### sigma 0 function: sigma0(x)

`sigma0` takes a 32-bit input and rotates its bits right by 7 and 18 positions. It also does a normal right shift by 3 positions. It then XORs the three results to get a final 32-bit value.

A normal right shift drops bits off the right side and fills the left side with zeros. Rotation does not.

Defined as:

$$\sigma_0^{\{256\}}(x) = \text{ROTR}^{7}(x) \oplus \text{ROTR}^{18}(x) \oplus \text{SHR}^{3}(x)$$

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.1.2, equation 4.6: defines sigma0.

Implementation:

In [15]:
def sigma0(x):
    """SHA-256 lowercase sigma 0 function."""
    # Force the input to 32 bits
    x = np.uint32(x)

    # XOR the two rotations with a plain right shift by 3
    return ROTR(x, 7) ^ ROTR(x, 18) ^ (x >> np.uint32(3))

Tests:

In [16]:
x = np.uint32(0x12345678)

# Check each operation on its own
assert ROTR(x, 7) == np.uint32(0xF02468AC)
assert ROTR(x, 18) == np.uint32(0x159E048D)
assert (x >> np.uint32(3)) == np.uint32(0x02468ACF)

# Check the final sigma0 result
assert sigma0(x) == np.uint32(0xE7FCE6EE)

# A zero input should give zero
assert sigma0(np.uint32(0)) == np.uint32(0)

# Check the result type is still uint32
assert isinstance(sigma0(x), np.uint32)

# Print the result as 8 hex digits
print(f"sigma0: 0x{sigma0(x):08x}")
print("All sigma0 tests passed.")

sigma0: 0xe7fce6ee
All sigma0 tests passed.


`sigma0` is used when the message schedule grows from 16 words to 64 words. It helps build new schedule words from words already computed.

It is used in this calculation:

$$W_t = \sigma_1^{\{256\}}(W_{t-2}) + W_{t-7} + \sigma_0^{\{256\}}(W_{t-15}) + W_{t-16}$$

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §6.2.2: the SHA-256 hash computation, where the message schedule is built.

### sigma 1 function: sigma1(x)

`sigma1` takes a 32-bit input and rotates its bits right by 17 and 19 positions. It also does a normal right shift by 10 positions. It then XORs the three results to get a final 32-bit value.

Defined as:

$$\sigma_1^{\{256\}}(x) = \text{ROTR}^{17}(x) \oplus \text{ROTR}^{19}(x) \oplus \text{SHR}^{10}(x)$$

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.1.2, equation 4.7: defines sigma1.

Implementation:

In [17]:
def sigma1(x):
    """SHA-256 lowercase sigma 1 function."""
    # Force the input to 32 bits
    x = np.uint32(x)

    # XOR the two rotations with a plain right shift by 10
    return ROTR(x, 17) ^ ROTR(x, 19) ^ (x >> np.uint32(10))

Tests:

In [18]:
x = np.uint32(0x12345678)

# Check each operation on its own
assert ROTR(x, 17) == np.uint32(0x2B3C091A)
assert ROTR(x, 19) == np.uint32(0x8ACF0246)
assert (x >> np.uint32(10)) == np.uint32(0x00048D15)

# Check the final sigma1 result
assert sigma1(x) == np.uint32(0xA1F78649)

# A zero input should give zero
assert sigma1(np.uint32(0)) == np.uint32(0)

# Check the result type is still uint32
assert isinstance(sigma1(x), np.uint32)

# Print the result as 8 hex digits
print(f"sigma1: 0x{sigma1(x):08x}")
print("All sigma1 tests passed.")

sigma1: 0xa1f78649
All sigma1 tests passed.


`sigma1` is used when the SHA-256 message schedule grows from 16 words to 64 words. It combines words already computed to build new ones.

It is used in this calculation:

$$W_t = \sigma_1^{\{256\}}(W_{t-2}) + W_{t-7} + \sigma_0^{\{256\}}(W_{t-15}) + W_{t-16}$$

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §6.2.2: the SHA-256 hash computation, where the message schedule is built.

## Problem 3: Generating the SHA-256 Constants

SHA-256 uses 64 round constants. They come from the fractional parts of the cube roots of the first 64 prime numbers.

The goal is to generate them with Python and NumPy, show them in hexadecimal, and compare them with the values in the Secure Hash Standard.

### Research

NIST FIPS 180-4, Section 4.2.2 defines the 64 SHA-256 round constants. They are the first 32 bits of the fractional parts of the cube roots of the first 64 primes.

The standard also lists the expected hex values, so I can check my results against them.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.2.2: defines the 64 round constants and lists their hex values.

### Generating the First 64 Prime Numbers

First I need a function that generates the first `n` prime numbers. A number is prime if it can only be divided by 1 and itself.

The function checks each number and tests for divisors up to its square root.

**References**

- [Python docs: `math.isqrt`](https://docs.python.org/3/library/math.html#math.isqrt): the integer square root, used as the upper limit for divisors.
- [Wikipedia: Trial division](https://en.wikipedia.org/wiki/Trial_division): why testing divisors up to the square root is enough.

In [19]:
import math

def primes(n):
    """
    Generate the first n prime numbers.

    Parameters:
        n (int): The number of prime numbers to generate.

    Returns:
        list: A list containing the first n prime numbers.
    """
    prime_list = []
    number = 2

    while len(prime_list) < n:
        is_prime = True

        # Only test divisors up to the square root of number
        for divisor in range(2, math.isqrt(number) + 1):
            if number % divisor == 0:
                is_prime = False
                break

        # No divisors found, so number is prime
        if is_prime:
            prime_list.append(number)

        number += 1

    return prime_list


# Generate the first 64 prime numbers
prime_list = primes(64)

# Test the function
assert len(prime_list) == 64
assert prime_list[:10] == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

# Check that no prime in the list has a divisor
for p in prime_list:
    assert all(p % d != 0 for d in range(2, math.isqrt(p) + 1))

# Print the list and a pass message
print(prime_list)
print("All prime number tests passed.")

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]
All prime number tests passed.


In [20]:
# Generate the first 64 prime numbers
prime_list = primes(64)

# Test the function
assert len(prime_list) == 64
assert prime_list[:10] == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]

# Check that no prime in the list has a divisor
for p in prime_list:
    assert all(p % d != 0 for d in range(2, math.isqrt(p) + 1))

# Print the list and a pass message
print(prime_list)
print("All prime number tests passed.")

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151, 157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229, 233, 239, 241, 251, 257, 263, 269, 271, 277, 281, 283, 293, 307, 311]
All prime number tests passed.


### Calculating the Cube Roots

Next I calculate the cube root of each of the 64 primes using NumPy's `cbrt()` function.

I convert the primes to `float64` values so the calculations use floating point numbers.

**References**

- [NumPy docs: `numpy.cbrt`](https://numpy.org/doc/stable/reference/generated/numpy.cbrt.html): element-wise cube root.

`float64` gives 64 bits of precision, with 53 bits for the significand. The cube roots of these primes are all small numbers, since their integer parts are less than 9. That leaves enough precision to work out the fractional parts and pull out the 32 bits needed. The comparison with the NIST constants confirms the results are correct.

**References**

- [Python docs: Floating-Point Arithmetic](https://docs.python.org/3/tutorial/floatingpoint.html): how `float64` values are stored and why they are approximations.

In [21]:
# Calculate the cube roots of the prime numbers
cube_roots = np.cbrt(np.array(prime_list, dtype=np.float64))

# Print the cube roots
print(cube_roots)

[1.25992105 1.44224957 1.70997595 1.91293118 2.22398009 2.35133469
 2.57128159 2.66840165 2.84386698 3.07231683 3.14138065 3.33222185
 3.44821724 3.50339806 3.60882608 3.75628575 3.89299642 3.93649718
 4.0615481  4.14081775 4.1793392  4.29084043 4.36207067 4.4647451
 4.59470089 4.65700951 4.68754815 4.7474594  4.77685618 4.83458813
 5.0265257  5.07875308 5.15513674 5.18010147 5.30145919 5.32507402
 5.39469071 5.46255557 5.50687845 5.57205466 5.63574079 5.65665283
 5.75896522 5.77899657 5.81864787 5.83827246 5.95334181 6.06412699
 6.1001702  6.11803317 6.15344949 6.20582179 6.22308425 6.30799355
 6.35786118 6.40695858 6.45531481 6.47127363 6.51868392 6.54991162
 6.56541443 6.6418522  6.74599671 6.77516895]


### Extracting the Fractional Parts

The next step is to pull out the fractional part of each cube root.

I do this by subtracting the floor of each value from the cube root itself.

**References**

- [NumPy docs: `numpy.floor`](https://numpy.org/doc/stable/reference/generated/numpy.floor.html): rounds each value down to the nearest whole number.

In [22]:
# Extract the fractional parts
fractional_parts = cube_roots - np.floor(cube_roots)

# Print the fractional parts
print(fractional_parts)

[0.25992105 0.44224957 0.70997595 0.91293118 0.22398009 0.35133469
 0.57128159 0.66840165 0.84386698 0.07231683 0.14138065 0.33222185
 0.44821724 0.50339806 0.60882608 0.75628575 0.89299642 0.93649718
 0.0615481  0.14081775 0.1793392  0.29084043 0.36207067 0.4647451
 0.59470089 0.65700951 0.68754815 0.7474594  0.77685618 0.83458813
 0.0265257  0.07875308 0.15513674 0.18010147 0.30145919 0.32507402
 0.39469071 0.46255557 0.50687845 0.57205466 0.63574079 0.65665283
 0.75896522 0.77899657 0.81864787 0.83827246 0.95334181 0.06412699
 0.1001702  0.11803317 0.15344949 0.20582179 0.22308425 0.30799355
 0.35786118 0.40695858 0.45531481 0.47127363 0.51868392 0.54991162
 0.56541443 0.6418522  0.74599671 0.77516895]


### Converting to 32-bit Unsigned Integers

The fractional parts are multiplied by $2^{32}$ to get the first 32 bits.

I use `np.floor()` to remove any decimal places and convert the results into NumPy `uint32` values.

**References**

- [NumPy docs: `ndarray.astype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html): converts an array to another type, here `uint32`.

In [23]:
# Convert fractional parts into 32-bit unsigned integers
round_constants = np.floor(fractional_parts * (2**32)).astype(np.uint32)

# Print the round constants
print(round_constants)

[1116352408 1899447441 3049323471 3921009573  961987163 1508970993
 2453635748 2870763221 3624381080  310598401  607225278 1426881987
 1925078388 2162078206 2614888103 3248222580 3835390401 4022224774
  264347078  604807628  770255983 1249150122 1555081692 1996064986
 2554220882 2821834349 2952996808 3210313671 3336571891 3584528711
  113926993  338241895  666307205  773529912 1294757372 1396182291
 1695183700 1986661051 2177026350 2456956037 2730485921 2820302411
 3259730800 3345764771 3516065817 3600352804 4094571909  275423344
  430227734  506948616  659060556  883997877  958139571 1322822218
 1537002063 1747873779 1955562222 2024104815 2227730452 2361852424
 2428436474 2756734187 3204031479 3329325298]


### Displaying the Constants in Hexadecimal

The constants are now generated. I can show them in hexadecimal using Python's formatting syntax.

Each constant is shown as an 8-digit hexadecimal value.

**References**

- [Python docs: Format Specification Mini-Language](https://docs.python.org/3/library/string.html#format-specification-mini-language): the `08x` format used to print 8 hex digits.

In [24]:
# Display all 64 constants in hexadecimal
for i, constant in enumerate(round_constants):
    # Print the index and the constant as 8 hex digits
    print(f"K[{i:02}] = 0x{constant:08x}")

K[00] = 0x428a2f98
K[01] = 0x71374491
K[02] = 0xb5c0fbcf
K[03] = 0xe9b5dba5
K[04] = 0x3956c25b
K[05] = 0x59f111f1
K[06] = 0x923f82a4
K[07] = 0xab1c5ed5
K[08] = 0xd807aa98
K[09] = 0x12835b01
K[10] = 0x243185be
K[11] = 0x550c7dc3
K[12] = 0x72be5d74
K[13] = 0x80deb1fe
K[14] = 0x9bdc06a7
K[15] = 0xc19bf174
K[16] = 0xe49b69c1
K[17] = 0xefbe4786
K[18] = 0x0fc19dc6
K[19] = 0x240ca1cc
K[20] = 0x2de92c6f
K[21] = 0x4a7484aa
K[22] = 0x5cb0a9dc
K[23] = 0x76f988da
K[24] = 0x983e5152
K[25] = 0xa831c66d
K[26] = 0xb00327c8
K[27] = 0xbf597fc7
K[28] = 0xc6e00bf3
K[29] = 0xd5a79147
K[30] = 0x06ca6351
K[31] = 0x14292967
K[32] = 0x27b70a85
K[33] = 0x2e1b2138
K[34] = 0x4d2c6dfc
K[35] = 0x53380d13
K[36] = 0x650a7354
K[37] = 0x766a0abb
K[38] = 0x81c2c92e
K[39] = 0x92722c85
K[40] = 0xa2bfe8a1
K[41] = 0xa81a664b
K[42] = 0xc24b8b70
K[43] = 0xc76c51a3
K[44] = 0xd192e819
K[45] = 0xd6990624
K[46] = 0xf40e3585
K[47] = 0x106aa070
K[48] = 0x19a4c116
K[49] = 0x1e376c08
K[50] = 0x2748774c
K[51] = 0x34b0bcb5
K[52] = 0x39

### Comparing with the Secure Hash Standard

Finally I compare the generated constants with the 64 values listed in NIST FIPS 180-4.

If the generated values match the standard, the assertion should pass.

**References**

- [FIPS 180-4](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), §4.2.2: the source of the expected constants.
- [NumPy docs: `numpy.array_equal`](https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html): checks two arrays have the same shape and values.

In [25]:
# Constants taken from NIST FIPS 180-4, Section 4.2.2
expected_constants = [
    0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5,
    0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5,
    0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3,
    0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174,
    0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc,
    0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da,
    0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7,
    0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967,
    0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13,
    0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85,
    0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3,
    0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070,
    0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5,
    0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3,
    0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208,
    0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2
]

# Store the expected constants as uint32
expected_constants = np.array(expected_constants, dtype=np.uint32)

# Compare the generated constants with the standard
assert len(round_constants) == 64
assert np.array_equal(round_constants, expected_constants)

# Print a pass message
print("All 64 SHA-256 constants match the NIST standard!")

All 64 SHA-256 constants match the NIST standard!


### Conclusion

All 64 SHA-256 round constants were generated from the cube roots of the first 64 primes.

The generated constants match the values in NIST FIPS 180-4, which confirms the implementation is correct.